# Q, K, V 차원 뻥튀기와 축소 이해하기

NanoGPT의 `CausalSelfAttention`에서 B(Batch), T(Time), C(Channel/Embedding) 차원이 어떻게 변하는지 복습해 봅시다.

## 0. 기초 개념: B, T, C와 Q, K, V의 관계 및 역할

본격적인 차원 계산에 앞서, 각 기호가 의미하는 바와 코드 내에서의 역할을 정리해 봅니다.

**[데이터의 형태: B, T, C]**
* **`B` (Batch Size)**: 한 번에 병렬로 연산하는 문장(시퀀스)의 개수입니다.
* **`T` (Time / Sequence Length)**: 문장의 길이, 즉 시퀀스 내 토큰의 개수입니다.
* **`C` (Channel / n_embd)**: 토큰 하나를 표현하는 임베딩 벡터의 크기(차원 수)입니다.
* 👉 **코드에서의 역할**: 모델에 입력되는 데이터 `x`의 기본 형태 `(B, T, C)`를 구성하며, 모델의 모든 레이어를 통과하는 기준 규격이 됩니다.

**[어텐션의 핵심: Q, K, V]**
Self-Attention은 각 단어가 문맥을 파악하기 위해 다른 단어들과 정보를 교환하는 과정입니다.
* **`Q` (Query - 질문)**: "나는 이런 문맥적 정보가 필요해" (정보를 찾는 주체)
* **`K` (Key - 열쇠)**: "나는 이런 특성의 정보를 가지고 있어" (질문에 대답할 단서)
* **`V` (Value - 실제 값)**: "내가 전달해 줄 실제 알맹이(의미)는 이거야"
* 👉 **관계 및 코드에서의 역할**:
  1. **Score 계산**: `Q`와 `K`를 내적(점곱)하여 두 단어가 얼마나 연관되어 있는지(Attention Score)를 구합니다.
  2. **정보 융합**: 이 연관성(Score) 비율만큼 `V`를 가중합(Weighted Sum)하여 문맥이 반영된 새로운 단어 텐서를 만듭니다.
  3. **코드 구현**: 이 세 개를 따로 계산하지 않고, `c_attn = nn.Linear(C, 3 * C)` 계층을 사용해 `x`를 한 번에 `(B, T, 3C)` 차원으로 뻥튀기합니다. 그 다음 `split` 함수로 3등분하여 각각 Q, K, V로 나누어 씁니다. 이는 GPU 연산 효율성을 극대화하기 위함입니다.

---

### 1. 차원 뻥튀기 (Expansion)
* **입력 차원**: `(B, T, C)`
* **선형 계층 (`c_attn`)**: `nn.Linear(C, 3 * C)`
* **연산 후**: 한 번의 행렬 곱으로 Q, K, V를 동시에 구하기 위해 `(B, T, 3 * C)`로 뻥튀기 됩니다.

### 2. Q, K, V 분리 (Split)
* **연산**: `.split(C, dim=2)`
* **결과**: `(B, T, 3 * C)`가 각각 `(B, T, C)` 차원을 갖는 `q`, `k`, `v`로 쪼개집니다.

### 3. 멀티 헤드 어텐션 구조로 변경 (Reshape & Transpose)
* **차원 분리**: `C`를 헤드 개수(`n_head`)와 헤드당 차원(`hs = C // n_head`)으로 나눕니다. -> `(B, T, n_head, hs)`
* **전치 (Transpose)**: `transpose(1, 2)`를 적용하여 각 헤드별 연산이 가능하도록 `(B, n_head, T, hs)` 형태로 바꿉니다.

### 4. 다시 합치기 (Re-assemble)
* 연산 완료 후 차원은 여전히 `(B, n_head, T, hs)` 입니다.
* **원상 복구**: 다시 `.transpose(1, 2)`를 적용하여 `(B, T, n_head, hs)`로 바꾸고, `.view(B, T, C)`로 `n_head`와 `hs`를 곱해 원래 차원 `(B, T, C)`로 합칩니다.

## 이해도 평가 (Self-Test)

아래의 셀들에 있는 빈칸(`0` 또는 `None`으로 표기됨)을 알맞은 값이나 변수(B, T, C, n_head 등)로 변경한 후 실행하면서 이해도를 점검해 보세요.

In [ ]:
import torch
import torch.nn as nn

# [문제 1] GPT2-small 기준 설정
B = 4          # Batch size
T = 128        # Sequence length
C = 768        # Embedding dimension (n_embd)
n_head = 12    # Number of attention heads

# 1-1. 각 헤드의 차원(hs)은 얼마일까요?
hs = 0 # <-- 여기에 올바른 식을 적어주세요 (예: C // n_head)
print(f"Head size (hs): {hs}")

# 1-2. 초기 입력 x의 텐서 형태(Shape)는 무엇일까요?
x_shape = (0, 0, 0) # <-- B, T, C 변수들을 이용해 채워보세요.
print(f"Input x shape expected: {x_shape}")

In [ ]:
# [문제 2] 차원 뻥튀기 (Expansion)
# q, k, v를 한 번에 구하기 위한 선형 계층입니다.
c_attn = nn.Linear(C, 3 * C)

# 임의의 입력 x 생성
x = torch.randn(B, T, C)

# 2-1. c_attn을 통과한 직후 결과(qkv)의 텐서 형태(Shape)는 무엇일까요?
qkv = c_attn(x)
qkv_shape_expected = (0, 0, 0) # <-- 예상되는 형태를 적어보세요.

print(f"실제 qkv shape: {tuple(qkv.shape)}")
print(f"예상 qkv shape: {qkv_shape_expected}")

In [ ]:
# [문제 3] Split 및 멀티 헤드 차원 변경
q, k, v = qkv.split(C, dim=2)

# 3-1. q를 쪼개고 전치(transpose)하기 직전, view를 거친 후의 차원은 무엇일까요?
q_view_shape_expected = (0, 0, 0, 0) # <-- (B, T, n_head, hs) 순서로 맞춰보세요.

# 3-2. 최종적으로 transpose(1, 2)를 거친 후 q의 최종 텐서 형태(Shape)는 무엇일까요?
q_final = q.view(B, T, n_head, C // n_head).transpose(1, 2)
q_final_shape_expected = (0, 0, 0, 0) # <-- 정답을 적어보세요.

print(f"실제 최종 q shape: {tuple(q_final.shape)}")
print(f"예상 최종 q shape: {q_final_shape_expected}")

In [ ]:
# [문제 4] Attention 결과 합치기 (Re-assemble)
# 어텐션 연산 결과 y가 만들어졌다고 가정합니다.
y = torch.randn(B, n_head, T, C // n_head)

# 4-1. 원래 형태 (B, T, C)로 되돌리기 위해 빈칸(0으로 표기됨)을 올바른 인자들로 채워보세요.
# 정답을 작성 후 주석(#)을 해제하고 실행해보세요!
# y_reassembled = y.transpose(0, 0).contiguous().view(0, 0, 0)

# print(f"복구된 y shape: {tuple(y_reassembled.shape)}")